# DeepDenoiser Training — ISTerre Event Rescue Pipeline

**Goal**: fine-tune Zhu et al. (2019) DeepDenoiser on a chosen event class's waveforms
(ice quake, rockslide, or any future class), using signal/noise pairs prepared by
`03c_denoiser_event_data.py` (the `EVENT_TYPE` constant at the top of that script
picks the class).

**Drive folder expected for GoogleColab** (name it after the event class to avoid collisions between
runs for different classes, e.g. `colab_deepdenoiser_training_rockslide`):
```
MyDrive/colab_deepdenoiser_training_<event_slug>/
    deepdenoiser/   <- Zhu code
    signal/         <- signal .npz files (GOOD population for that class)
    noise/          <- noise .npz files
    signal_list.csv
    noise_list.csv
```

**Runtime on GoogleColab**: Runtime > Change runtime type > T4 GPU

## Cell 1 — Check GPU

In [8]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

Mon Jul  6 13:04:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Mount Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## Cell 3 — Configure paths
**Edit this cell** to match your Drive folder

In [ ]:
import os

# ── Edit here ──────────────────────────────────────────────────────────
# EVENT_TYPE should match the EVENT_TYPE used in 03c_denoiser_event_data.py for the run that produced the signal/noise files
EVENT_TYPE       = 'rockslide'
EVENT_SLUG       = EVENT_TYPE.lower().replace(' ', '_')
DRIVE_BASE       = f'/content/drive/MyDrive/colab_deepdenoiser_training_{EVENT_SLUG}'
DEEPDENOISER_DIR = os.path.join(DRIVE_BASE, 'deepdenoiser')
SIGNAL_DIR       = os.path.join(DRIVE_BASE, 'signal')
NOISE_DIR        = os.path.join(DRIVE_BASE, 'noise')
SIGNAL_LIST      = os.path.join(DRIVE_BASE, 'signal_list.csv')
NOISE_LIST       = os.path.join(DRIVE_BASE, 'noise_list.csv')
LOG_DIR          = os.path.join(DRIVE_BASE, 'log')
EPOCHS        = 50
BATCH_SIZE    = 20
LEARNING_RATE = 0.001
# ───────────────────────────────────────────────────────────────────────

os.makedirs(LOG_DIR, exist_ok=True)
for label, path in [('deepdenoiser/', DEEPDENOISER_DIR), ('signal/', SIGNAL_DIR),
                    ('noise/', NOISE_DIR), ('signal_list', SIGNAL_LIST), ('noise_list', NOISE_LIST)]:
    exists = os.path.exists(path)
    count  = f'  ({len(os.listdir(path))} files)' if exists and os.path.isdir(path) else ''
    print(f"{'OK' if exists else 'MISSING'}  {label:20s}  {path}{count}")


## Cell 4 — Check TensorFlow + GPU visibility

In [11]:
import tensorflow as tf
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {gpus}')
if not gpus:
    print('WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU')

TensorFlow : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Cell 4b — Install missing dependencies

`obspy` is not pre-installed on Colab. `tf-keras` provides the legacy Keras 2 backend required
by Zhu's TF1-era code (`tf.compat.v1.layers`). Re-run after every runtime restart.

In [12]:
import subprocess, sys
for pkg in ['obspy', 'tf-keras']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print(f"[{'OK' if r.returncode == 0 else 'FAILED'}] {pkg}")
import obspy, tf_keras
print(f'obspy {obspy.__version__}  /  tf_keras {tf_keras.__version__}')

[OK] obspy
[OK] tf-keras
obspy 1.5.0  /  tf_keras 2.20.0


## Cell 4c — Fix .npz channels format

`03c_denoiser_event_data.py` saved `channels` as `np.bytes_`. Zhu's DataReader calls
`.tolist()` on it giving Python bytes `b'GU.BLANC'`, which never matches the plain string
`'GU.BLANC'` in the noise CSV. This cell converts every .npz to Unicode strings.

Also detects and skips **corrupted files** (0 bytes or truncated) instead of crashing on the first one,
then drops any corrupted filenames out of `signal_list.csv` / `noise_list.csv` so `train.py` never tries to load them.

**Safe to re-run** — skips files that are already fixed

In [ ]:
import numpy as np, os, zipfile
import pandas as pd

def fix_channels_folder(folder, label):
    """
    Convert channels field to Unicode strings.
    
    Returns
    -------
    corrupted : list of str — filenames that could not be read
    """
    files = [f for f in os.listdir(folder) if f.endswith('.npz')]
    fixed = already_ok = 0
    corrupted = []
    for fname in files:
        fpath = os.path.join(folder, fname)
        if os.path.getsize(fpath) == 0:
            corrupted.append(fname)
            continue
        try:
            with np.load(fpath, allow_pickle=True) as d:
                channels_raw = d['channels']
                data = d['data'].copy()
                itp  = d['itp'].copy() if 'itp' in d else None
        except (EOFError, zipfile.BadZipFile, OSError, ValueError, KeyError) as e:
            corrupted.append(fname)
            continue
        if channels_raw.dtype.kind == 'S':
            channels_fixed = channels_raw.item().decode('utf-8')
            if itp is not None:
                np.savez(fpath, data=data, itp=itp, channels=channels_fixed)
            else:
                np.savez(fpath, data=data, channels=channels_fixed)
            fixed += 1
        else:
            already_ok += 1
    print(f'{label:12s}: fixed {fixed:3d}   already OK {already_ok:3d}   '
          f'corrupted {len(corrupted):3d}   total {len(files)}')
    if corrupted:
        print(f'  [WARN] Corrupted/unreadable {label} files (skipped):')
        for f in corrupted:
            print(f'    {f}')
    return corrupted

def drop_corrupted_from_list(list_path, corrupted):
    """Remove rows referencing corrupted filenames from a signal/noise list CSV."""
    if not corrupted:
        return
    df = pd.read_csv(list_path)
    before = len(df)
    df = df[~df['fname'].isin(corrupted)]
    df.to_csv(list_path, index=False)
    print(f'  [FIXED] {os.path.basename(list_path)}: dropped {before - len(df)} row(s) '
          f'-> {len(df)} remain')

corrupted_signal = fix_channels_folder(SIGNAL_DIR, 'signal/')
corrupted_noise  = fix_channels_folder(NOISE_DIR,  'noise/')

drop_corrupted_from_list(SIGNAL_LIST, corrupted_signal)
drop_corrupted_from_list(NOISE_LIST,  corrupted_noise)

print('Done.')


## Cell 4d — Patch data_reader.py on Drive

Patches two incompatibilities in Zhu's original code **directly in the Drive file**:
- `np.complex_` removed in NumPy 2.0 (3 occurrences) → `np.complex128`
- `snr_threshold = 10` rejects most Alpine events (calibrated for California broadband
  earthquakes) → set to 0, i.e. disable the filter so all confirmed events of the target class are used for training

**Safe to re-run** — the assertions verify the result. Run once; change persists on Drive.

In [ ]:
import os

dr_path = os.path.join(DEEPDENOISER_DIR, 'data_reader.py')
with open(dr_path, 'r') as f:
    code = f.read()

code = code.replace('dtype=np.complex_)',
                    'dtype=np.complex128)  # np.complex_ removed in NumPy 2.0')
code = code.replace('snr_threshold = 10',
                    'snr_threshold = 0   # Zhu default (10) rejects most Alpine events; disabled so all confirmed events of the target class are used')

with open(dr_path, 'w') as f:
    f.write(code)

assert 'np.complex_)' not in code, 'np.complex_ still present!'
assert 'snr_threshold = 10' not in code, 'snr_threshold = 10 still present!'
print('Patched OK.')
print(f'  np.complex128 occurrences : {code.count("np.complex128")}')
print(f'  snr_threshold lines       : {[l.strip() for l in code.splitlines() if "snr_threshold =" in l and "config" not in l]}')


## Cell 4e — Drop orphan-station signal rows (no matching noise)

**Safe to re-run** — no-op if there are no orphan stations


In [ ]:
import pandas as pd

sig_df   = pd.read_csv(SIGNAL_LIST)
noise_df = pd.read_csv(NOISE_LIST)

orphan_stations = set(sig_df['channels']) - set(noise_df['channels'])

if orphan_stations:
    n_before = len(sig_df)
    sig_df = sig_df[~sig_df['channels'].isin(orphan_stations)]
    sig_df.to_csv(SIGNAL_LIST, index=False)
    print(f'[FIXED] {len(orphan_stations)} orphan station(s), dropped '
          f'{n_before - len(sig_df)} row(s) -> {len(sig_df)} remain in signal_list.csv')
    for st in sorted(orphan_stations):
        print(f'  {st}')
else:
    print('[OK] Every station in signal_list.csv has at least one matching noise window.')


## Cell 5 — Run training

Expected time on T4 GPU: ~15-30 min for 50 epochs.
`TF_USE_LEGACY_KERAS=1` forces Keras 2 backend so `tf.compat.v1.layers` works.

In [15]:
import sys, subprocess, os

train_script = os.path.join(DEEPDENOISER_DIR, 'train.py')
cmd = [
    sys.executable, train_script,
    '--train_signal_dir',  SIGNAL_DIR,
    '--train_signal_list', SIGNAL_LIST,
    '--train_noise_dir',   NOISE_DIR,
    '--train_noise_list',  NOISE_LIST,
    '--log_dir',           LOG_DIR,
    '--epochs',            str(EPOCHS),
    '--batch_size',        str(BATCH_SIZE),
    '--learning_rate',     str(LEARNING_RATE),
]

env = os.environ.copy()
env['TF_USE_LEGACY_KERAS'] = '1'  # force Keras 2 for tf.compat.v1.layers

print('Command:', ' '.join(cmd))
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nTraining finished.' if proc.returncode == 0 else f'\nExited with code {proc.returncode}.')

Command: /usr/bin/python3 /content/drive/MyDrive/colab_deepdenoiser_training/deepdenoiser/train.py --train_signal_dir /content/drive/MyDrive/colab_deepdenoiser_training/signal --train_signal_list /content/drive/MyDrive/colab_deepdenoiser_training/signal_list.csv --train_noise_dir /content/drive/MyDrive/colab_deepdenoiser_training/noise --train_noise_list /content/drive/MyDrive/colab_deepdenoiser_training/noise_list.csv --log_dir /content/drive/MyDrive/colab_deepdenoiser_training/log --epochs 50 --batch_size 20 --learning_rate 0.001
/content/drive/MyDrive/colab_deepdenoiser_training/deepdenoiser/util.py:58: SyntaxWarning: invalid escape sequence '\h'
  plt.title("$\hat{Y}$")
/content/drive/MyDrive/colab_deepdenoiser_training/deepdenoiser/util.py:131: SyntaxWarning: invalid escape sequence '\h'
  plt.title("$\hat{Y}$")
2026-07-06 13:04:50,438 Dataset size: training 263, validation 0
2026-07-06 13:04:50,438 Training log: /content/drive/MyDrive/colab_deepdenoiser_training/log/260706-130450

## Cell 6 — Find the checkpoint path

Copy the path printed here and paste it as `MODEL_DIR` in `03c_denoiser_event_data.py`

In [ ]:
import glob
ckpt_folders = sorted(glob.glob(os.path.join(LOG_DIR, '*')))
if ckpt_folders:
    latest = ckpt_folders[-1]
    print(f'Latest checkpoint folder:\n  {latest}')
    print(f'\nSet in 03c_denoiser_event_data.py:')
    print(f'  MODEL_DIR = "{latest}"')
    print(f'\nFiles:')
    for f in sorted(os.listdir(latest)):
        print(f'  {f}')
else:
    print('No checkpoint folder found — did training complete?')
